# Бенчмарк `_fwhm_interp`: старая реализация (`np.interp`) vs новая (скан от вершины)

Ячейки по порядку: реализации → синтетическая нагрузка → проверка эквивалентности → `%timeit` → масштабирование по числу пиков.

Запускать повторно безопасно: нумба кэширует компиляцию (`cache=True`), синтетика детерминирована по seed.

In [1]:
import time
import numpy as np
from numba import njit, prange

rng_global = np.random.default_rng(0)

## Реализации

Старая — как в `_peakpicker_core`: `np.interp` по всему окну склона, правый склон через реверс срезов.

Новая — ручной скан от вершины до первого пересечения half-max + та же линейная интерполяция между двумя точками.

In [2]:
@njit(cache=True)
def _fwhm_interp_old(mz, intens, pmz_array, pint_array, left_min, right_min):
    n_peaks = pmz_array.size
    fwhm_left = np.empty(n_peaks, dtype=np.float64)
    fwhm_right = np.empty(n_peaks, dtype=np.float64)
    for n in range(n_peaks):
        lm = left_min[n]
        rm = right_min[n]
        vm = pint_array[n]
        pp = pmz_array[n]
        half_max = vm / 2.0
        fwhm_left[n] = np.interp(half_max, intens[lm:pp + 1], mz[lm:pp + 1])
        fwhm_right[n] = np.interp(half_max,
                                  intens[pp:rm + 1][::-1],
                                  mz[pp:rm + 1][::-1])
    return fwhm_right - fwhm_left


@njit(cache=True)
def _fwhm_interp_new(mz, intens, pmz_array, pint_array, left_min, right_min):
    n_peaks = pmz_array.size
    fwhm = np.empty(n_peaks, dtype=np.float64)
    for n in range(n_peaks):
        pp = pmz_array[n]
        lm = left_min[n]
        rm = right_min[n]
        half = pint_array[n] * 0.5

        # левый склон: от вершины влево до первого пересечения half
        fwl = mz[lm]
        for i in range(pp - 1, lm - 1, -1):
            if intens[i] <= half:
                y0, y1 = intens[i], intens[i + 1]
                x0, x1 = mz[i], mz[i + 1]
                t = (half - y0) / (y1 - y0) if y1 != y0 else 0.0
                fwl = x0 + t * (x1 - x0)
                break

        # правый склон: зеркально
        fwr = mz[rm]
        for i in range(pp + 1, rm + 1):
            if intens[i] <= half:
                y0, y1 = intens[i - 1], intens[i]
                x0, x1 = mz[i - 1], mz[i]
                t = (half - y0) / (y1 - y0) if y1 != y0 else 0.0
                fwr = x0 + t * (x1 - x0)
                break

        fwhm[n] = fwr - fwl
    return fwhm

## Синтетическая нагрузка

2000 гауссовых пиков шириной 2–6 точек на оси из 100k точек — реалистичный порядок для профиля МС. Границы пиков — вершина ± пара ширин, как в реальном пайплайне.

In [3]:
def make_bench_input(n_peaks=1500, n_points=100_000, seed=0):
    rng = np.random.default_rng(seed)
    mz = np.linspace(100.0, 1000.0, n_points)
    step = mz[1] - mz[0]

    centers = np.sort(rng.uniform(150.0, 950.0, n_peaks))
    # отбрасываем центры ближе min_gap друг к другу
    min_gap = step * 40          # с запасом перекрывает ±3σ хвосты и окна
    keep = [centers[0]]
    for c in centers[1:]:
        if c - keep[-1] >= min_gap:
            keep.append(c)
    centers = np.array(keep)

    widths_pts = rng.integers(2, 7, centers.size)
    amps = rng.uniform(100.0, 5000.0, centers.size)

    intens = np.zeros(n_points)
    for c, w, a in zip(centers, widths_pts, amps):
        i0 = int((c - mz[0]) / step)
        lo, hi = max(i0 - 3 * w, 0), min(i0 + 3 * w, n_points)
        x = mz[lo:hi]
        sigma = w * step / 2.35482
        intens[lo:hi] += a * np.exp(-0.5 * ((x - c) / sigma) ** 2)

    pmz = np.rint((centers - mz[0]) / step).astype(np.int64)
    half_w = widths_pts // 2 + 1
    left = np.clip(pmz - half_w, 0, n_points - 2).astype(np.int64)
    right = np.clip(pmz + half_w + 1, 1, n_points - 1).astype(np.int64)
    pint = intens[pmz].astype(np.float64)

    return mz, intens, pmz.astype(np.float64), pint, left, right


args2000 = make_bench_input(n_peaks=2000)
print(f"пиков: {args2000[2].size}, точек: {args2000[0].size}")

пиков: 1048, точек: 100000


## Проверка эквивалентности (до всяких замеров)

На монотонных склонах совпадение должно быть до машинного эпсилона. Расхождения > 1e-9 — остановиться и разобрать конкретные пики, прежде чем радоваться ускорению.

In [4]:
out_old = _fwhm_interp_old(*args2000)
out_new = _fwhm_interp_new(*args2000)

diff = np.abs(out_old - out_new)
print(f"max |diff| = {diff.max():.3e}")
print(f"расхождений > 1e-9: {(diff > 1e-9).sum()} из {diff.size}")
if (diff > 1e-9).any():
    bad = np.where(diff > 1e-9)[0][:10]
    print("первые расхождения, номера пиков:", bad)

max |diff| = 0.000e+00
расхождений > 1e-9: 0 из 1048


## `%timeit`: базовое сравнение

Первый вызов каждой функции уже случился выше (компиляция прошла). Замеряем чистое исполнение.

In [5]:
%timeit -n 20 -r 7 _fwhm_interp_old(*args2000)

497 μs ± 16.9 μs per loop (mean ± std. dev. of 7 runs, 20 loops each)


In [6]:
%timeit -n 20 -r 7 _fwhm_interp_new(*args2000)

14.2 μs ± 3.38 μs per loop (mean ± std. dev. of 7 runs, 20 loops each)


## С `prange` (parallel=True)

Пики независимы — параллелим по пикам. Отдельная функция, не трогаем ядро целиком.

In [7]:
@njit(cache=True, parallel=True)
def _fwhm_interp_par(mz, intens, pmz_array, pint_array, left_min, right_min):
    n_peaks = pmz_array.size
    fwhm = np.empty(n_peaks, dtype=np.float64)
    for n in prange(n_peaks):
        pp = pmz_array[n]
        lm = left_min[n]
        rm = right_min[n]
        half = pint_array[n] * 0.5

        fwl = mz[lm]
        for i in range(pp - 1, lm - 1, -1):
            if intens[i] <= half:
                y0, y1 = intens[i], intens[i + 1]
                x0, x1 = mz[i], mz[i + 1]
                t = (half - y0) / (y1 - y0) if y1 != y0 else 0.0
                fwl = x0 + t * (x1 - x0)
                break

        fwr = mz[rm]
        for i in range(pp + 1, rm + 1):
            if intens[i] <= half:
                y0, y1 = intens[i - 1], intens[i]
                x0, x1 = mz[i - 1], mz[i]
                t = (half - y0) / (y1 - y0) if y1 != y0 else 0.0
                fwr = x0 + t * (x1 - x0)
                break

        fwhm[n] = fwr - fwl
    return fwhm

In [8]:
out_par = _fwhm_interp_par(*args2000)
print(f"max |diff| vs new = {np.abs(out_par - out_new).max():.3e}")

max |diff| vs new = 0.000e+00


In [9]:
%timeit -n 20 -r 7 _fwhm_interp_par(*args2000)

34 μs ± 7.56 μs per loop (mean ± std. dev. of 7 runs, 20 loops each)


## Масштабирование по числу пиков

Где старая версия начинает проседать: она аллоцирует реверснутые срезы на каждый пик — при больших списках это основной источник расходов. Через `timeit.repeat` — устойчивая оценка (берём минимум).

In [10]:
import timeit

rows = []
for n_peaks in (100, 500, 2000, 10000):
    args = make_bench_input(n_peaks=n_peaks, seed=1)
    for name, fn in [("np.interp", _fwhm_interp_old),
                     ("scan     ", _fwhm_interp_new),
                     ("scan+par ", _fwhm_interp_par)]:
        fn(*args)  # прогрев/компиляция под новые типы — типы те же, но на всякий случай
        t = min(timeit.repeat(lambda: fn(*args), number=20, repeat=7)) / 20
        rows.append((n_peaks, name, t))

print(f"{'peaks':>7} | {'реализация':<10} | {'мс/вызов':>10}")
for n_peaks, name, t in rows:
    print(f"{n_peaks:>7} | {name:<10} | {t*1000:>10.3f}")

  peaks | реализация |   мс/вызов
    100 | np.interp  |      0.077
    100 | scan       |      0.005
    100 | scan+par   |      0.023
    500 | np.interp  |      0.315
    500 | scan       |      0.009
    500 | scan+par   |      0.017
   2000 | np.interp  |      0.647
   2000 | scan       |      0.012
   2000 | scan+par   |      0.013
  10000 | np.interp  |      0.874
  10000 | scan       |      0.021
  10000 | scan+par   |      0.017


In [11]:
# и наглядно: относительное ускорение
import collections
by_n = collections.defaultdict(dict)
for n_peaks, name, t in rows:
    by_n[n_peaks][name.strip()] = t
for n_peaks, d in by_n.items():
    base = d["np.interp"]
    print(f"n_peaks={n_peaks:>5}: scan x{base/d['scan']:.1f}, scan+par x{base/d['scan+par']:.1f}")

n_peaks=  100: scan x15.3, scan+par x3.4
n_peaks=  500: scan x37.0, scan+par x18.4
n_peaks= 2000: scan x54.4, scan+par x50.1
n_peaks=10000: scan x41.9, scan+par x51.4


## Что читать из результатов

- `scan` должен быть заметно быстрее уже на узких пиках (окно поиска крошечное, а аллокации реверснутых срезов исчезают).
- `scan+par` даёт выигрыш на тысячах пиков; на сотнях может быть вровень или хуже — оверхед планирования `prange`.
- Если ускорение скана на ваших данных мизерное — узкое место не здесь, и трогать функцию не стоит.

In [12]:
mz, intens, pmz, pint, left, right = args2000

for n in [43, 377, 412]:
    pp = int(pmz[n]); lm = int(left[n]); rm = int(right[n])
    half = pint[n] / 2.0
    print(f"\n=== пик {n}: apex@{pp}, окно [{lm}, {rm}], half={half:.2f}")

    for side, sl in (("левый", intens[lm:pp+1]), ("правый", intens[pp:rm+1])):
        if side == "левый":
            mono = np.all(np.diff(sl) >= -1e-12)
        else:
            mono = np.all(np.diff(sl) <= 1e-12)
        crossings = np.where(sl <= half)[0]
        print(f"{side}: монотонный={mono}, пересечений half={crossings.size}, "
              f"края: {sl[0]:.2f} … {sl[-1]:.2f}")
        if crossings.size > 1:
            print(f"  немонотонность: пересечения на смещениях {crossings[:5]}")

    print(f"old={out_old[n]:.5f}  new={out_new[n]:.5f}  diff={diff[n]:.3e}")


=== пик 43: apex@9326, окно [9322, 9331], half=546.84
левый: монотонный=True, пересечений half=2, края: 236.35 … 1093.68
  немонотонность: пересечения на смещениях [0 1]
правый: монотонный=True, пересечений half=2, края: 1093.68 … 231.96
  немонотонность: пересечения на смещениях [4 5]
old=0.05500  new=0.05500  diff=0.000e+00

=== пик 377: apex@37974, окно [37972, 37977], half=1399.87
левый: монотонный=True, пересечений half=2, края: 162.78 … 2799.75
  немонотонность: пересечения на смещениях [0 1]
правый: монотонный=True, пересечений half=2, края: 2799.75 … 6.09
  немонотонность: пересечения на смещениях [2 3]
old=0.01806  new=0.01806  diff=0.000e+00

=== пик 412: apex@40765, окно [40762, 40769], half=774.59
левый: монотонный=True, пересечений half=1, края: 535.80 … 1549.19
правый: монотонный=True, пересечений half=3, края: 1549.19 … 49.85
  немонотонность: пересечения на смещениях [2 3 4]
old=0.03745  new=0.03745  diff=0.000e+00
